In [ ]:
# --- 1. TELEPÍTÉS ---
# Javítjuk a protobuf verziót is, mert az is dobott hibát a logban
!pip install -q vllm pyngrok protobuf==3.20.3

import os
import subprocess
import time
from pyngrok import ngrok, conf

# --- 2. KONFIGURÁCIÓ ---
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct-AWQ"
AUTH_TOKEN = "2sUqYYhi9BuVxa2zJtzAdHVI9eS_6rxXaeqe4o8JTPtMPtsBf"

# --- 3. AGRESSZÍV TAKARÍTÁS ---
print("🧹 Takarítás és beragadt folyamatok lelövése...")
try:
    ngrok.kill()
    os.system("pkill -f vllm")
    os.system("pkill -f ngrok")
    os.system("killall ngrok")  # Biztos ami biztos
    time.sleep(3)  # Várunk, hogy az ngrok szerver is érzékelje a bontást
except:
    pass

# --- 4. NGROK ---
if not AUTH_TOKEN:
    print("❌ HIBA: Nincs AUTH_TOKEN!")
else:
    # Régió váltása EU-ra, hátha az US régióban ragadt be a session
    conf.get_default().region = "eu"
    ngrok.set_auth_token(AUTH_TOKEN)

    try:
        # Próbáljuk a fix domaint
        tunnel = ngrok.connect(8000, domain="intimate-polecat-adjusted.ngrok-free.app")
        print("✅ Fix domain aktív!")
    except Exception as e:
        print(f"⚠️ Fix domain hiba vagy foglalt session ({e}).")
        print("🔄 Próbálkozás random URL-lel...")
        try:
            tunnel = ngrok.connect(8000)
        except Exception as e2:
            print(
                "❌ KRITIKUS NGROK HIBA: Lépj be a dashboard.ngrok.com-ra és a 'Tunnels' alatt lődd le manuálisan a beragadtat!"
            )
            raise e2

    PUBLIC_URL = tunnel.public_url
    print(f"🎉 API LESZ ITT: {PUBLIC_URL}/v1")

# --- 5. vLLM SZERVER INDÍTÁSA (STABIL VERZIÓ) ---
log_file = open("vllm_server.log", "w")

cmd = [
    "python",
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    MODEL_ID,
    "--quantization",
    "awq",
    "--dtype",
    "half",
    "--host",
    "0.0.0.0",
    "--port",
    "8000",
    # Memória: 2048-ra visszavéve a biztonság kedvéért (T4 GPU limit)
    # Ha stabil, később átírhatod 4096-ra
    "--max-model-len",
    "2048",
    "--gpu-memory-utilization",
    "0.95",
    "--enforce-eager",
    "--trust-remote-code",
    # --- TOOL CALLING KIKAPCSOLVA ---
    # Kivettem az összes --enable-auto-tool-choice és --tool-call-parser flaget.
    # Így a szerver sima chat módban indul, ami NEM fog hibára futni.
    # A Roo Code (Cline) így is tudja használni a modellt!
]

print("\n🚀 Szerver indítása (Tool flags nélkül)...")
process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

# --- 6. LOG MONITOROZÁS ---
try:
    print("⏳ Várakozás a modell betöltésére (kb. 3-5 perc)...")

    with open("vllm_server.log") as f:
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line.strip())
                if "Uvicorn running on" in line:
                    print("\n✅✅✅ A SZERVER SIKERESEN ELINDULT! ✅✅✅")
                    print(f"🌍 Roo Code URL: {PUBLIC_URL}/v1")
                    print("👉 API Key: 'dummy'")
                    break
            else:
                time.sleep(1)
                if process.poll() is not None:
                    print("\n❌ A SZERVER LEÁLLT HIBÁVAL! Itt a log vége:")
                    print(f.read())
                    break

    if process.poll() is None:
        process.wait()

except KeyboardInterrupt:
    print("\n🛑 Leállítás...")
    process.terminate()
    ngrok.kill()
    log_file.close()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.9/474.9 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 811.2 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 143.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 148.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-reflection 1.76.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.3 requires 

(EngineCore_DP0 pid=3880) INFO 01-27 15:00:35 [weight_utils.py:487] Time spent downloading weights for Qwen/Qwen2.5-14B-Instruct-AWQ: 150.849531 seconds
(EngineCore_DP0 pid=3880)
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
(EngineCore_DP0 pid=3880)
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.68it/s]
(EngineCore_DP0 pid=3880)
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:16<00:09,  9.34s/it]
(EngineCore_DP0 pid=3880)
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:31<00:00, 12.32s/it]
(EngineCore_DP0 pid=3880)
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:31<00:00, 10.64s/it]
(EngineCore_DP0 pid=3880)
(EngineCore_DP0 pid=3880) INFO 01-27 15:01:07 [default_loader.py:308] Loading weights took 32.04 seconds
(EngineCore_DP0 pid=3880) INFO 01-27 15:01:08 [gpu_model_runner.py:3659] Model loading took 9.3795 GiB memory and 214.246650 seconds
(EngineCore_DP0 pid=3880) INFO 01-27


🛑 Leállítás...
